In [1]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

import os

In [2]:
#from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama

# Initialize the "Brain" (LLM)
llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0)

In [3]:
from langchain_core.tools import tool

# 1. Define the Calculator Tool
@tool
def add(a: int, b: int) -> int:
    """ Adds two integers and returns the result."""
    return a + b

# 1. Define the Calculator Tool
@tool
def multiply(a: int, b: int) -> int:
    """ Multiply two integers and returns the result."""
    return a * b

# 2. Define a "Mock" Search Tool (Simulating Real-time data)
@tool
def get_stock_price(ticker: str) -> str:
    """Returns the current stock price for a given ticker symbol. 
    Use 'ZENSAR' for Zensar Technologies, 'GOOGL' for Google."""
    # In a real agent, this would call Yahoo Finance API
    ticker_upper = ticker.upper().strip()
    if "ZENSAR" in ticker_upper:
        return "464.00 INR"
    elif "GOOGL" in ticker_upper or "GOOGLE" in ticker_upper:
        return "175.00 USD"
    else:
        return f"Unknown Ticker: {ticker}"

# List of tools available to our Agent
#tools = [multiply, get_stock_price]

print("Tools Created: Calculator (Hands) & Stock Search (Eyes)")

Tools Created: Calculator (Hands) & Stock Search (Eyes)


In [4]:
#from tavily import TavilyClient

#tavily_client = TavilyClient(api_key="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")
#response = tavily_client.search(query="Who is Leo Messi?",max_results=3)

#print(response)

In [5]:


os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

In [6]:
from langchain_community.tools.tavily_search import TavilySearchResults

C:\Users\sr43993\AppData\Local\Temp\ipykernel_27324\723937994.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [7]:
tavily_tool = TavilySearchResults(max_results=3)

C:\Users\sr43993\AppData\Local\Temp\ipykernel_27324\2877639967.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=3)


In [8]:
#response = tavily_tool.invoke("Who is Leo Messi?")

#response

In [9]:
from langchain.agents import create_agent

# Create a ReAct agent using the modern LangChain API
# debug=True enables verbose output showing the Thought -> Action -> Observation loop
#agent_executor = create_agent(llm, tools, debug=True)
agent_executor = create_agent(
    model=llm,
    tools=[multiply, add, get_stock_price, tavily_tool ],
    system_prompt="You are a helpful assistant you only provide the answers based on the tools ,don't use your own knowledge and don't make up any answer if you don't have the answer in the tools, if you don't have the answer in the tools then say I don't know",
)


print("Agent created successfully with ReAct capabilities!")

Agent created successfully with ReAct capabilities!


In [10]:
response = agent_executor.invoke({"messages": [("user", "What is 49823 * 19283?")]})
response

{'messages': [HumanMessage(content='What is 49823 * 19283?', additional_kwargs={}, response_metadata={}, id='3efee791-27aa-4120-8c4b-a6612e06d7fa'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-06-19T10:22:53.26780079Z', 'done': True, 'done_reason': 'stop', 'total_duration': 899634837, 'load_duration': None, 'prompt_eval_count': 326, 'prompt_eval_duration': None, 'eval_count': 55, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'}, id='lc_run--019edf67-99f3-7850-8e63-701a3b232b0d-0', tool_calls=[{'name': 'multiply', 'args': {'a': 49823, 'b': 19283}, 'id': '63bf6315-bdf0-4b5c-9ddf-26cdb30b9b8a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 326, 'output_tokens': 55, 'total_tokens': 381}),
  ToolMessage(content='960736909', name='multiply', id='f883f6b6-a22e-456e-a9c2-9418225c45a7', tool_call_id='63bf6315-bdf0-4b5c-9ddf-26cdb30b9b8a'),
  AIMessa

In [11]:
#print(f"\nFinal Answer: {response['messages'][-1].content}")